In [3]:
import random
import numpy as np
import sympy as sp
import h5py
import torch
import torch.nn as nn
from typing import List, Dict, Tuple, Union, Optional

# Class to handle the tokenizing of an amplitude

## Definition

In [3]:
class ScatteringAmplitudeTokenizer:
    """
    Tokenizer for scattering amplitude expressions involving momenta p_i, 
    polarization vectors e_i, and field strengths F.
    """
    def __init__(self, max_particles: int = 8, max_sequence_length: int = 2048): # 8 particles by default
        self.max_particles = max_particles
        self.max_sequence_length = max_sequence_length
        
        # Define initial hard-coded vocabulary (added "0:" for digit-level encoding)
        self.vocab_init = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<BOS>": 2,
            "<EOS>": 3,
            "+": 4,
            "-": 5,
            "*": 6,
            "/": 7,
            "^": 8,
            "(": 9,
            ")": 10,
            "0:": 11,
            "1:": 12,
            "2:": 13,
            "3:": 14,
            "4:": 15,
            "5:": 16,
            "6:": 17,
            "7:": 18,
            "8:": 19,
            "9:": 20,
            "10:": 21,
            "·": 22,  # dot operation    
        }
        
        self.index = len(self.vocab_init)
        
        # Momentum tokens: p_1, p_2, ..., p_n
        self.p_tokens = {f"p_{i}": i + self.index for i in range(1, max_particles + 1)}
        
        # Polarization tokens: e_1, e_2, ..., e_n
        self.e_tokens = {f"e_{i}": i + self.index + max_particles for i in range(1, max_particles + 1)}
        
        # Field strength tokens: F_1, F_2, ..., F_n
        self.f_tokens = {f"F_{i}": i + self.index + 2 * max_particles for i in range(1, max_particles + 1)}
        
        # Build the full vocabulary
        self.vocab = {**self.vocab_init, **self.p_tokens, **self.e_tokens, **self.f_tokens}
        
        # Inverse mapping
        self.id_to_token = {v: k for k, v in self.vocab.items()}
        
        self.vocab_size = len(self.vocab)
    
    def tokenize(self, expression: str) -> List[int]:
        """
        Convert a string representation of a scattering amplitude expression to tokens.
        
        Example: "p_1·p_2 + e_1·p_2 + p_1·F_3·p_2"
        """
        # Preprocessing: Normalize the expression
        expression = expression.replace(" ", "")
        
        # Split the expression at operators while keeping them
        parts = []
        current = ""
        for char in expression:
            if char in "+-*()^":
                if current:
                    parts.append(current)
                parts.append(char)
                current = ""
            else:
                current += char
        if current:
            parts.append(current)
        
        # Process each term
        tokens = [self.vocab["<BOS>"]]
        
        for part in parts:
            if part in "+-*()^":
                tokens.append(self.vocab[part])
            else:
                # Check if the part is a number
                if part.isdigit():
                    # If number is between 1 and 10, use the pre-defined token
                    if 1 <= int(part) <= 10:
                        tokens.append(self.vocab[f"{part}:"])
                    else:
                        # For numbers outside our vocabulary, split into digit tokens.
                        for digit in part:
                            key = f"{digit}:"
                            if key in self.vocab:
                                tokens.append(self.vocab[key])
                            else:
                                tokens.append(self.vocab["<UNK>"])
                else:
                    # Handle dot product terms like "p_1·p_2" or "p_1·F_3·p_2"
                    subparts = part.split("·")
                    for i, subpart in enumerate(subparts):
                        if subpart in self.vocab:
                            tokens.append(self.vocab[subpart])
                        else:
                            # Try to match with known patterns
                            found = False
                            for prefix in ["p_", "e_", "F_"]:
                                if subpart.startswith(prefix):
                                    idx = subpart[len(prefix):]
                                    token_key = f"{prefix}{idx}"
                                    if token_key in self.vocab:
                                        tokens.append(self.vocab[token_key])
                                        found = True
                                        break
                            if not found:
                                tokens.append(self.vocab["<UNK>"])
                        
                        # Add dot token between subparts
                        if i < len(subparts) - 1:
                            tokens.append(self.vocab["·"])
        
        tokens.append(self.vocab["<EOS>"])
        
        # Ensure the sequence doesn't exceed max length
        if len(tokens) > self.max_sequence_length:
            tokens = tokens[:self.max_sequence_length - 1] + [self.vocab["<EOS>"]]
        
        return tokens
    
    def pad_sequence(self, tokens: List[int]) -> List[int]:
        """Pad or truncate a token sequence to max_sequence_length"""
        if len(tokens) < self.max_sequence_length:
            tokens = tokens + [self.vocab["<PAD>"]] * (self.max_sequence_length - len(tokens))
        else:
            tokens = tokens[:self.max_sequence_length]
        return tokens
    
    def batch_encode(self, expressions: List[str]) -> torch.Tensor:
        """Tokenize and pad a batch of expressions"""
        batch_tokens = [self.pad_sequence(self.tokenize(expr)) for expr in expressions]
        return torch.tensor(batch_tokens, dtype=torch.long)
    
    def decode(self, tokens: List[int]) -> str:
        """Convert tokens back to a string expression with reduced whitespace."""
        # Convert token ids to strings (strip trailing colons for digit tokens)
        decoded = []
        for token in tokens:
            if token in self.id_to_token and token not in [self.vocab["<PAD>"], 
                                                        self.vocab["<BOS>"], 
                                                        self.vocab["<EOS>"]]:
                token_str = self.id_to_token[token]
                if token_str.endswith(":") and token_str[:-1].isdigit():
                    decoded.append(token_str[:-1])
                else:
                    decoded.append(token_str)
        
        # Merge consecutive digit tokens into single numbers
        merged_tokens = []
        number_buffer = ""
        for tok in decoded:
            if tok.isdigit():
                number_buffer += tok
            else:
                if number_buffer:
                    merged_tokens.append(number_buffer)
                    number_buffer = ""
                merged_tokens.append(tok)
        if number_buffer:
            merged_tokens.append(number_buffer)
        
        # Join tokens with a single space and remove spaces around dot operator, multiplication and power
        expression = " ".join(merged_tokens)
        expression = expression.replace(" · ", "·").replace(" * ", "*").replace(" ^ ", "^").replace("( ", "(").replace(" )", ")")
        
        return expression.strip()



## Test

In [4]:
tokenizer = ScatteringAmplitudeTokenizer()
print(tokenizer.vocab)
print(tokenizer.vocab_size)

# Example expressions
expressions = [
    "p_1·p_2",
    "e_1·p_2",
    "e_1·e_2",
    "p_1·F_3·p_2",
    "p_1·F_3·F_4·p_2",
    "p_1·F_3·F_4·F_5·p_2",
    "(p_1·p_2 + 3*e_1·p_2 + 123*p_1·F_3·p_2)^124",
    "(8*p_1·p_2 + 3*e_1·p_2 + 57*p_1·F_3·F_2·F_3·F_4·p_2)^8",
]

# Tokenize examples
for expr in expressions:
    tokens = tokenizer.tokenize(expr)
    decoded = tokenizer.decode(tokens)
    print(f"Original: {expr}")
    print(f"Tokens: {tokens}")
    print(f"Decoded: {decoded}")
    print()

{'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3, '+': 4, '-': 5, '*': 6, '/': 7, '^': 8, '(': 9, ')': 10, '0:': 11, '1:': 12, '2:': 13, '3:': 14, '4:': 15, '5:': 16, '6:': 17, '7:': 18, '8:': 19, '9:': 20, '10:': 21, '·': 22, 'p_1': 24, 'p_2': 25, 'p_3': 26, 'p_4': 27, 'p_5': 28, 'p_6': 29, 'p_7': 30, 'p_8': 31, 'e_1': 32, 'e_2': 33, 'e_3': 34, 'e_4': 35, 'e_5': 36, 'e_6': 37, 'e_7': 38, 'e_8': 39, 'F_1': 40, 'F_2': 41, 'F_3': 42, 'F_4': 43, 'F_5': 44, 'F_6': 45, 'F_7': 46, 'F_8': 47}
47
Original: p_1·p_2
Tokens: [2, 24, 22, 25, 3]
Decoded: p_1·p_2

Original: e_1·p_2
Tokens: [2, 32, 22, 25, 3]
Decoded: e_1·p_2

Original: e_1·e_2
Tokens: [2, 32, 22, 33, 3]
Decoded: e_1·e_2

Original: p_1·F_3·p_2
Tokens: [2, 24, 22, 42, 22, 25, 3]
Decoded: p_1·F_3·p_2

Original: p_1·F_3·F_4·p_2
Tokens: [2, 24, 22, 42, 22, 43, 22, 25, 3]
Decoded: p_1·F_3·F_4·p_2

Original: p_1·F_3·F_4·F_5·p_2
Tokens: [2, 24, 22, 42, 22, 43, 22, 44, 22, 25, 3]
Decoded: p_1·F_3·F_4·F_5·p_2

Original: (p_1·p_2 + 3*e_1·p_2 + 

# Class for dot products

## Definition

COMMENTS:
- discuss if I called functions within a class correctly, eg DP.distribute_DP
- to_token_list function inside DP obsolete?

In [4]:
class DP(sp.Function):
    @classmethod
    def eval(cls, a, b):
        # No automatic simplification
        return None

    def _pretty(self, printer, **kwargs):
        # For pretty printing (used by sp.pprint).
        a_form = printer._print(self.args[0])
        b_form = printer._print(self.args[1])
        #dot_form = prettyForm("⋅")  # Unicode centered dot

        # Concatenate the pretty-printed components
        return a_form*b_form
    
    def extract_scalar_factor(expr):
        """
        Given a sympy expression 'expr', return (c, remainder) so that
            expr == c * remainder
        where c is the numeric coefficient (including its sign) and
        remainder is the non-numeric part.
        """
        c, args = expr.as_coeff_mul()
        remainder = sp.Mul(*args) if args else sp.Integer(1)
        return (c, remainder)
    
    def distribute_DP(expr):
        """
        Recursively distribute the DP dot product over additions in its arguments,
        and factor out any numeric coefficients so that DP(-p1, p2) becomes
        -DP(p1, p2).
        
        For example:
        DP(-p1, p2 + 2*p3)  ->  -DP(p1, p2) - 2*DP(p1, p3)
        """
        # Base case: if there's no DP or if the expression is atomic, return it unchanged.
        if not expr.has(DP) or expr.is_Atom:
            return expr

        # Handle the DP instance explicitly.
        if expr.func == DP:
            a, b = expr.args
            # Recursively expand the sub-arguments.
            a_expanded = DP.distribute_DP(a)
            b_expanded = DP.distribute_DP(b)
            
            # Factor out numeric coefficients from each argument.
            ca, a_noscale = DP.extract_scalar_factor(a_expanded)
            cb, b_noscale = DP.extract_scalar_factor(b_expanded)
            c_total = ca * cb
            dp_core = DP(a_noscale, b_noscale)
            
            # Distribute over sums if present in either argument.
            if a_noscale.is_Add and b_noscale.is_Add:
                terms = [DP.distribute_DP(DP(term_a, term_b))
                        for term_a in a_noscale.args
                        for term_b in b_noscale.args]
                return c_total * sp.Add(*terms)
            elif a_noscale.is_Add:
                terms = [DP.distribute_DP(DP(term, b_noscale))
                        for term in a_noscale.args]
                return c_total * sp.Add(*terms)
            elif b_noscale.is_Add:
                terms = [DP.distribute_DP(DP(a_noscale, term))
                        for term in b_noscale.args]
                return c_total * sp.Add(*terms)
            else:
                # No sum to distribute over; simply return the DP multiplied by the scalar.
                return c_total * dp_core

        # Otherwise, apply the distribution recursively to the subexpressions.
        new_args = [DP.distribute_DP(arg) for arg in expr.args]
        return expr.func(*new_args)
    
    def to_token_list(self):
        """
        Return a token list for this DP expression (or any Sympy expression).
        This method uses a recursive tree traversal.
        """
        return tokenize_expr(self)

## Test

In [17]:
test = DP(sp.Symbol("a")+sp.Symbol("c"),sp.Symbol("b"))
print(test)

distest = DP.distribute_DP(test)
print(distest)

DP(a + c, b)
DP(a, b) + DP(c, b)


# Class to generate data

## Definition

COMMENTS:
- discuss if I called functions within a class correctly, eg ScatteringAmplitudeGenerator.generate_monomial and ScatteringAmplitudeGenerator.generate_denominator

In [124]:
class NewAmplitudeGenerator:
    """
    Generate training data for scattering amplitude expressions.
    Each data point will be a tuple of input and target strings.
    The input string will be expressed in terms of momenta p_i and polarization vectors e_i.
    The target string will be expressed in terms of momenta p_i and field strengths F_i.
    """
    def __init__(self, max_particles: int = 8, max_dim: int = 10):
        self.max_particles = max_particles
        self.max_dim = max_dim
        
    def generate_monomial(self, n, n_gluons, n_gravitons, dim):
        """
        Generate a Sympy expression of total mass-dimension 'dim'.
        """
        
        # For gluons: exactly 1 copy of each label i in 1..n_gluons
        # For gravitons: 2 copies of each label j in n_gluons+1..n_gluons+n_gravitons. We assume that graviton polarization tensors are factorized into vectors.
        pol_indices = []
        for i in range(1, n_gluons + 1):
            pol_indices.append(i)
        for j in range(n_gluons + 1, n_gluons + n_gravitons + 1):
            pol_indices.append(j)
            pol_indices.append(j)
        
        total_pols = len(pol_indices)  # = n_gluons + 2*n_gravitons
        
        
        # We need to make sure the amplitude ends up having the correct mass dimension, noting that
        #   [pi.pj] = 2, [ei.pj] = 1, [ei.ej] = 0
        # The mass dimension (without coupling constants) is then given by
        #   dim = n(ei.pj) + n(pi.pj) = x + 2y
        # where x is the number of ei.pj factors and y is the number of pi.pj factors.
        # If we define z = n(ei.ej), then we have the constraints
        #   x + 2z = total_pols
        #   x + 2y = dim
        # This gives
        #   y = z + (dim - total_pols)/2
        # Recall that z is n(ei.ej), so it must be non-negative, an integer, and have a maximum total_pols//2.
        # 
        # This means we won't always have a valid solution to the above equation! In that case, we'll just return a trivial expression "1". Or maybe we should issue a warning?
        # Either way, we will try and solve this for z meeting the above constraints, and randomly choose a solution if there are more than 1.
        # We will also demand that x,y,z are all positive as we're trying to construct the amplitude numerator. The denominator is a seperate process.
        
        valid_solutions = []
        for z_candidate in range(total_pols//2 + 1):  # z can go up to total_pols//2
            x_candidate = total_pols - 2*z_candidate
            
            if x_candidate < 0:  # We want positive x
                continue
            # From y = z + (dim - total_pols)/2, we also need (dim - total_pols) to be even for y to be an integer
            shift = (dim - total_pols)
            if shift % 2 != 0:
                # We don't want fractional y values so we ignore any odd shifts!
                continue
            y_candidate = z_candidate + shift//2
            if y_candidate < 0: # We also want positive y
                continue
            # Checks passed - we have a valid triple (x_candidate, y_candidate, z_candidate)
            valid_solutions.append((x_candidate, y_candidate, z_candidate))
        
        # If no valid solutions, return a trivial expression "1". Maybe issue a warning here that no valid numerators?
        if not valid_solutions:
            return "1"
        
        # Choose a random solution for variety
        x, y, z = random.choice(valid_solutions)
        
        # Now we have: a set of gluon and graviton pol_indices, and how many of each type of factor we should include to satisfy the mass dimension constraints.
        # We now want to generate random factors that satisfy these constraints.
        # We'll:
        #   (a) Shuffle pol_indices, then pick x of them for e_i.p_j factors,
        #   (b) from the remainder, group them in pairs to form e_i.e_j factors,
        #   (c) build y p_i.p_j factors by picking random pairs of momentum indices,
        #   (d) shuffle them all, then form a product.

        factors = []
        
        # Shuffle and pick x polarizations for e_i.p_j
        random.shuffle(pol_indices)
        
        ep_indices = pol_indices[:x]   # e_i's that will go with p_j
        remaining  = pol_indices[x:]   # leftover for e_i.e_j
        # Annoyingly, after shuffling, these indices can have repeated indices side-by-side, meaning we could generate ei.ei terms (at least for gravitons). Need to figure out a way to avoid this.
        
        # build the e_i.p_j factors
        for e_idx in ep_indices:
            # choose a random momentum index in [1..n] that is not the same as e_idx
            momentum_choices = [mn for mn in range(1, n+1) if mn != e_idx]
            m_idx = random.choice(momentum_choices)
            e_sym = f"e{e_idx}"
            p_sym = f"p{m_idx}"
            factors.append(f"{e_sym}.{p_sym}") # polarizations to the left!
        
        # From the remaining polarization indices, pair them up for e_i.e_j
        # (We must have exactly 2z = len(remaining) from the valid solution.)

        for i in range(z):
            #print("Remaining:", remaining)
            # We need to remove any side-by-side duplicates to avoid ei.ei terms
            # Check for duplicates
            for _ in range(50):  # Try up to 50 times de-shuffle the remaining indices. In testing, never need more than 3 or 4 iterations.
                for j in range(0, len(remaining), 2):
                        if remaining[j] == remaining[j+1]:
                            if len(remaining) == 2:
                                print(f"Remaining: {remaining}. Impossible to avoid duplicate e_i.e_i terms. Quitting...")
                                return "1"
                            else:
                                print("Duplicate in e_i.e_j:", remaining, ", shuffling.")
                                valid = False
                                break
                        else:
                            valid = True
                if valid:
                    break
                    #print("No duplicates in e_i.e_j:", remaining)
                else:
                    valid = True
                    random.shuffle(remaining)
            
            i1 = remaining[2*i]
            i2 = remaining[2*i + 1]
            e1 = f"e{i1}"
            e2 = f"e{i2}"
            factors.append(f"{e1}.{e2}")

        # Now build the factors of p_i.p_j to ensure the correct mass dimension
        # We just pick y random pairs of momentum indices in [1..n] and dot them together.
        for _ in range(y):
            i1 = random.randint(1, n)
            i2 = random.choice([mn for mn in range(1, n+1) if mn != i1])
            p1 = f"p{i1}"
            p2 = f"p{i2}"
            factors.append(f"{p1}.{p2}")
        
        # Shuffle all factors for a bit of spice and multiply them into a single expression
        random.shuffle(factors)
        if not factors:
            return "1"
        
        monomial = " * ".join(factors) if factors else "1"  # multiply all factors together
        
        return monomial
    
    def generate_denominator(self, n, dim):
        """
        For now, we only consider momentum in the denominator (Mandelstams, essenially), so 
        we generate a product of d/2 momentum dot products.
        
        """
        denom = []
        for _ in range(dim//2): # We want im/2 pairs of momentum indices. dim should be even!
            i1 = random.randint(1, n)
            i2 = random.choice([mn for mn in range(1, n+1) if mn != i1])
            p1 = f"p{i1}"
            p2 = f"p{i2}"
            denom.append(f"{p1}.{p2}")
        random.shuffle(denom)
        return " * ".join(denom)
    
    def generate_amplitude(self, n, n_gluons, n_gravitons, dim, max_monomials=3):
        """
        Generate a random amplitude with n external particles (n_gluons gluons and n_gravitons gravitons), mass-dimension dim, with up to max_monomials monomials.
        
        We will choose the denominator have mass dimension 2(n-3) and therefore the numerator to have mass dimension dim + 2(n-3). 
        """
        i = random.choice(range(1, max_monomials+1))
        
        monos = []
        for _ in range(1, i+1):
            monos.append(self.generate_monomial(n, n_gluons, n_gravitons, dim + 2*(n-3)))
        #print("Monomials:", monos)
        numerator = "+".join(monos)
        denominator = self.generate_denominator(n, 2*(n-3))
        return f"({numerator})/{denominator}"

In [6]:
class ScatteringAmplitudeGenerator:
    """
    Generate training data for scattering amplitude expressions.
    Each data point will be a tuple of input and target strings.
    The input string will be expressed in terms of momenta p_i and polarization vectors e_i.
    The target string will be expressed in terms of momenta p_i and field strengths F_i.
    """
    def __init__(self, max_particles: int = 8, max_dim: int = 10):
        self.max_particles = max_particles
        self.max_dim = max_dim
        
    def generate_monomial(self, n, n_gluons, n_gravitons, dim):
        """
        Generate a Sympy expression of total mass-dimension 'dim'.
        """
        
        # For gluons: exactly 1 copy of each label i in 1..n_gluons
        # For gravitons: 2 copies of each label j in n_gluons+1..n_gluons+n_gravitons. We assume that graviton polarization tensors are factorized into vectors.
        pol_indices = []
        for i in range(1, n_gluons + 1):
            pol_indices.append(i)
        for j in range(n_gluons + 1, n_gluons + n_gravitons + 1):
            pol_indices.append(j)
            pol_indices.append(j)
        
        total_pols = len(pol_indices)  # = n_gluons + 2*n_gravitons
        
        
        # We need to make sure the amplitude ends up having the correct mass dimension, noting that
        #   [pi.pj] = 2, [ei.pj] = 1, [ei.ej] = 0
        # The mass dimension (without coupling constants) is then given by
        #   dim = n(ei.pj) + n(pi.pj) = x + 2y
        # where x is the number of ei.pj factors and y is the number of pi.pj factors.
        # If we define z = n(ei.ej), then we have the constraints
        #   x + 2z = total_pols
        #   x + 2y = dim
        # This gives
        #   y = z + (dim - total_pols)/2
        # Recall that z is n(ei.ej), so it must be non-negative, an integer, and have a maximum total_pols//2.
        # 
        # This means we won't always have a valid solution to the above equation! In that case, we'll just return a trivial expression "1". Or maybe we should issue a warning?
        # Either way, we will try and solve this for z meeting the above constraints, and randomly choose a solution if there are more than 1.
        # We will also demand that x,y,z are all positive as we're trying to construct the amplitude numerator. The denominator is a seperate process.
        
        valid_solutions = []
        for z_candidate in range(total_pols//2 + 1):  # z can go up to total_pols//2
            x_candidate = total_pols - 2*z_candidate
            
            if x_candidate < 0:  # We want positive x
                continue
            # From y = z + (dim - total_pols)/2, we also need (dim - total_pols) to be even for y to be an integer
            shift = (dim - total_pols)
            if shift % 2 != 0:
                # We don't want fractional y values so we ignore any odd shifts!
                continue
            y_candidate = z_candidate + shift//2
            if y_candidate < 0: # We also want positive y
                continue
            # Checks passed - we have a valid triple (x_candidate, y_candidate, z_candidate)
            valid_solutions.append((x_candidate, y_candidate, z_candidate))
        
        # If no valid solutions, return a trivial expression "1". Maybe issue a warning here that no valid numerators?
        if not valid_solutions:
            return sp.Integer(1)
        
        # Choose a random solution for variety
        x, y, z = random.choice(valid_solutions)
        
        # Now we have: a set of gluon and graviton pol_indices, and how many of each type of factor we should include to satisfy the mass dimension constraints.
        # We now want to generate random factors that satisfy these constraints.
        # We'll:
        #   (a) Shuffle pol_indices, then pick x of them for e_i.p_j factors,
        #   (b) from the remainder, group them in pairs to form e_i.e_j factors,
        #   (c) build y p_i.p_j factors by picking random pairs of momentum indices,
        #   (d) shuffle them all, then form a product.

        factors = []
        
        # Shuffle and pick x polarizations for e_i.p_j
        random.shuffle(pol_indices)
        
        ep_indices = pol_indices[:x]   # e_i's that will go with p_j
        remaining  = pol_indices[x:]   # leftover for e_i.e_j
        # Annoyingly, after shuffling, these indices can have repeated indices side-by-side, meaning we could generate ei.ei terms (at least for gravitons). Need to figure out a way to avoid this.
        
        # build the e_i.p_j factors
        for e_idx in ep_indices:
            # choose a random momentum index in [1..n] that is not the same as e_idx
            momentum_choices = [mn for mn in range(1, n+1) if mn != e_idx]
            m_idx = random.choice(momentum_choices)
            e_sym = sp.Symbol(f"e{e_idx}", commutative=True)
            p_sym = sp.Symbol(f"p{m_idx}", commutative=True)
            factors.append(DP(e_sym, p_sym)) # polarizations to the left!
        
        # From the remaining polarization indices, pair them up for e_i.e_j
        # (We must have exactly 2z = len(remaining) from the valid solution.)

        for i in range(z):
            #print("Remaining:", remaining)
            # We need to remove any side-by-side duplicates to avoid ei.ei terms
            # Check for duplicates
            for _ in range(50):  # Try up to 50 times de-shuffle the remaining indices. In testing, never need more than 3 or 4 iterations.
                for j in range(0, len(remaining), 2):
                        if remaining[j] == remaining[j+1]:
                            print("Duplicate in e_i.e_j:", remaining, ", shuffling.")
                            valid = False
                            break
                        else:
                            valid = True
                if valid:
                    break
                    #print("No duplicates in e_i.e_j:", remaining)
                else:
                    valid = True
                    random.shuffle(remaining)
            
            i1 = remaining[2*i]
            i2 = remaining[2*i + 1]
            e1 = sp.Symbol(f"e{i1}", commutative=True)
            e2 = sp.Symbol(f"e{i2}", commutative=True)
            factors.append(DP(e1, e2))

        # Now build the factors of p_i.p_j to ensure the correct mass dimension
        # We just pick y random pairs of momentum indices in [1..n] and dot them together.
        for _ in range(y):
            i1 = random.randint(1, n)
            i2 = random.choice([mn for mn in range(1, n+1) if mn != i1])
            p1 = sp.Symbol(f"p{i1}", commutative=True)
            p2 = sp.Symbol(f"p{i2}", commutative=True)
            factors.append(DP(p1, p2))
        
        # Shuffle all factors for a bit of spice and multiply them into a single expression
        random.shuffle(factors)
        if not factors:
            return sp.Integer(1)
        
        monomial = sp.Mul(*factors)  # multiply all factors together
        
        return monomial
    
    def generate_denominator(self, n, dim):
        """
        For now, we only consider momentum in the denominator (Mandelstams, essenially), so 
        we generate a product of d/2 momentum dot products.
        
        """
        denom = []
        for _ in range(dim//2): # We want im/2 pairs of momentum indices. dim should be even!
            i1 = random.randint(1, n)
            i2 = random.choice([mn for mn in range(1, n+1) if mn != i1])
            p1 = sp.Symbol(f"p{i1}", commutative=True)
            p2 = sp.Symbol(f"p{i2}", commutative=True)
            denom.append(DP(p1, p2))
        random.shuffle(denom)
        return sp.Mul(*denom)
    
    def generate_amplitude(self, n, n_gluons, n_gravitons, dim, max_monomials=3):
        """
        Generate a random amplitude with n external particles (n_gluons gluons and n_gravitons gravitons), mass-dimension dim, with up to max_monomials monomials.
        
        We will choose the denominator have mass dimension 2(n-3) and therefore the numerator to have mass dimension dim + 2(n-3). 
        """
        i = random.choice(range(1, max_monomials+1))
        
        monos = []
        for _ in range(1, i+1):
            monos.append(self.generate_monomial(n, n_gluons, n_gravitons, dim + 2*(n-3)))
        #print("Monomials:", monos)
        numerator = sp.Add(*monos)
        denominator = self.generate_denominator(n, 2*(n-3))
        return numerator/denominator

In [162]:
class NewAmplitudeScrambler:
    """
    Generate training data for scattering amplitude expressions.
    Each data point will be a tuple of input and target strings.
    The input string will be expressed in terms of momenta p_i and polarization vectors e_i.
    The target string will be expressed in terms of momenta p_i and field strengths F_i.
    """
    def __init__(self, max_particles: int = 8, max_dim: int = 10, num_scrambles: int = 3):
        self.max_particles = max_particles
        self.max_dim = max_dim
        self.num_scrambles = num_scrambles

    @staticmethod
    def momentum_conservation_substitution(j, n):
        """
        Substitute "p{j}" with "-(p1 + p2 + ... + p{j-1} + p{j+1} + ... + p{n})" in string expressions.
        """
        replacement_terms = [f"p{i}" for i in range(1, n+1) if i != j]
        replacement = "-(" + " + ".join(replacement_terms) + ")"
        return {f"p{j}": replacement}

    def single_scramble(self, expr, n, n_gluons, n_gravitons):
        """
        Perform ONE random scramble operation on 'expr' (a string):
        
        1) Multiply by 1 using (e_i.p_j)/(e_i.p_j), applying momentum conservation in the numerator.
        2) Add zero using e_i.p_i = 0 and momentum conservation on p_i in the numerator.
        3) Directly apply momentum conservation to some p_j in the entire expression.
        
        If n_gluons == 0 and n_gravitons == 0, we only perform operation 3.
        """
        # Choose the operation
        if n_gluons == 0 and n_gravitons == 0:
            operation = 3
        else:
            operation = random.choice([1, 2, 3])
        
        total_polarizations = n_gluons + n_gravitons

        if operation == 1:
            # --- Operation 1: Multiply by 1 ---
            # Choose random polarization index (for e_i) and momentum index (for p_j)
            i = random.randint(1, total_polarizations)
            j = random.randint(1, n)
            e_i = f"e{i}"
            p_j = f"p{j}"
            
            # Get the momentum conservation replacement for p_j
            subs = NewAmplitudeScrambler.momentum_conservation_substitution(j, n)
            replacement = subs[p_j]
            
            # Build the numerator as e_i dot (replacement) and denominator as e_i dot p_j.
            num_expr = f"{e_i}.({replacement})"
            den_expr = f"{e_i}.{p_j}"
            
            # The factor is (num_expr)/(den_expr), which is mathematically 1.
            factor = f"(({num_expr})/({den_expr}))"
            return f"(({expr})*{factor})"
        
        elif operation == 2:
            # --- Operation 2: Add zero ---
            # Choose a polarization index i, and then choose a momentum index j
            i = random.randint(1, total_polarizations)
            if i <= n:
                possible_js = [x for x in range(1, n+1) if x != i]
                if not possible_js:
                    possible_js = list(range(1, n+1))
            else:
                possible_js = list(range(1, n+1))
            j = random.choice(possible_js)
            
            e_i = f"e{i}"
            p_i = f"p{i}"
            p_j = f"p{j}"
            
            # Apply momentum conservation to p_i if applicable
            if i <= n:
                subs = NewAmplitudeScrambler.momentum_conservation_substitution(i, n)
                replacement = subs[p_i]
            else:
                replacement = p_i

            # The zero term is built from e_i dot (momentum-conserved p_i) divided by e_i dot p_j.
            zero_num = f"{e_i}.({replacement})"
            den_expr = f"{e_i}.{p_j}"
            zero_term = f"(({zero_num})/({den_expr}))"
            
            # To add fractions, force the original expression to have the same denominator.
            expr_as_fraction = f"(({expr})*({den_expr}/{den_expr}))"
            return f"(({expr_as_fraction}) + ({zero_term}))"
        
        else:  # operation == 3
            # --- Operation 3: Direct momentum conservation ---
            # Choose a random momentum index j and replace all occurrences of p_j in the expression.
            j = random.randint(1, n)
            p_j = f"p{j}"
            subs = NewAmplitudeScrambler.momentum_conservation_substitution(j, n)
            replacement = subs[p_j]
            # Replace every occurrence of p_j with the momentum-conserved expression.
            scrambled_expr = expr.replace(p_j, f"({replacement})")
            return scrambled_expr

    def scramble(self, amp, n, n_gluons=0, n_gravitons=0, only_scramble_numerator=True):
        """
        Repeatedly scramble the amplitude 'amp' (a string) using the three operations.
        
        Parameters:
          amp: Amplitude expression as a string (e.g., "((numerator))/(denom)").
          n: Number of external momenta (p1, p2, ..., p_n).
          n_gluons, n_gravitons: Numbers used to select polarization indices.
          only_scramble_numerator: If True, apply the scramble operations only to the numerator.
        """
        scrambled_amp = amp
        for _ in range(self.num_scrambles):
            if only_scramble_numerator:
                # Assume the amplitude is in the form "(numerator)/(denom)".
                try:
                    # Split only on the first occurrence of "/".
                    num, den = scrambled_amp.split("/", 1)
                    num = num.strip(" ()")
                    den = den.strip(" ()")
                except Exception:
                    num, den = scrambled_amp, "1"
                new_num = self.single_scramble(num, n, n_gluons, n_gravitons)
                scrambled_amp = f"({new_num})/({den})"
            else:
                scrambled_amp = self.single_scramble(scrambled_amp, n, n_gluons, n_gravitons)
        return scrambled_amp


In [7]:
class ScatteringAmplitudeScrambler:
    """
    Generate training data for scattering amplitude expressions.
    Each data point will be a tuple of input and target strings.
    The input string will be expressed in terms of momenta p_i and polarization vectors e_i.
    The target string will be expressed in terms of momenta p_i and field strengths F_i.
    """
    def __init__(self, max_particles: int = 8, max_dim: int = 10, num_scrambles: int = 3):
        self.max_particles = max_particles
        self.max_dim = max_dim
        self.num_scrambles = num_scrambles
    
    def momentum_conservation_substitution(j, n):
        """
        Substitute p_j = - sum_{k != j}^n p_k.
        
        """
        p_syms = [sp.Symbol(f"p{i}", commutative=True) for i in range(1, n+1)] # sympy symbols for p1, p2, ..., p_n
        replacement = -sum(p_syms[k] for k in range(n) if (k != (j-1)))
        return {p_syms[j-1]: replacement}
    

    def single_scramble(self, expr, n, n_gluons, n_gravitons):
        """
        Perform ONE random scramble operation on 'expr':
        1) Multiply by 1 using (e_i.p_j)/(e_i.p_j), applying momentum
            conservation in the numerator only.
        2) Add zero using e_i.p_i=0 and momentum conservation on p_i in the numerator.
        3) Directly apply momentum conservation to some p_j in the entire expression.

        If n_gluons==0 and n_gravitons==0, we only do operation #3 (scalars).
        """
        # If no polarizations, skip 1 and 2:
        if n_gluons == 0 and n_gravitons == 0:
            # Operation 3 only
            operation = 3
        else:
            operation = random.choice([1, 2, 3])
        
        total_polarizations = n_gluons + n_gravitons
        
        if operation == 1:
            # --- Multiply by 1 using ( e_i . p_j ) / ( e_i . p_j ) ---
            i = random.randint(1, total_polarizations)
            j = random.randint(1, n)

            e_i = sp.Symbol(f"e{i}", commutative=True)
            p_j = sp.Symbol(f"p{j}", commutative=True)

            # Numerator with momentum conservation on p_j
            subs_dict = ScatteringAmplitudeScrambler.momentum_conservation_substitution(j, n)
            num_expr = DP.distribute_DP(DP(e_i, p_j).subs(subs_dict))
            den_expr = DP(e_i, p_j)

            return expr * (num_expr/den_expr)

        elif operation == 2:
            # --- Add zero using e_i.p_i=0, then momentum conservation on p_i ---
            i = random.randint(1, total_polarizations)

            # We want a momentum index j != i only if i <= n. Otherwise any j in 1..n is fine.
            # But let's handle corner cases gracefully.
            if i <= n:
                possible_js = [x for x in range(1, n+1) if x != i]
                if not possible_js:
                    possible_js = list(range(1, n+1))  # fallback
            else:
                possible_js = list(range(1, n+1))

            j = random.choice(possible_js)

            e_i = sp.Symbol(f"e{i}", commutative=True)
            p_i = sp.Symbol(f"p{i}", commutative=True)  # only valid if i <= n
            p_j = sp.Symbol(f"p{j}", commutative=True)

            # ( e_i . p_i ), with p_i replaced by sum_{k != i}, then / ( e_i . p_j )
            subs_dict = {}
            if i <= n:
                subs_dict = ScatteringAmplitudeScrambler.momentum_conservation_substitution(i, n)

            zero_num_subbed = DP.distribute_DP(DP(e_i, p_i).subs(subs_dict))
            zero_term = zero_num_subbed / DP(e_i, p_j)

            # Combine over common denominator e_i . p_j
            common_denom = DP(e_i, p_j)
            expr_as_fraction = expr * (common_denom / common_denom)

            return expr_as_fraction + zero_term

        else:
            # --- Operation 3: Direct momentum conservation on some p_j in the entire expression ---
            j = random.randint(1, n)
            subs_dict = ScatteringAmplitudeScrambler.momentum_conservation_substitution(j, n)
            return DP.distribute_DP(expr.subs(subs_dict))
        
    def scramble(self, amp, n, n_gluons=0, n_gravitons=0, only_scramble_numerator=True):
        """
        Repeatedly scramble the amplitude 'amp' using the three operations.
        
        Parameters:
        amp : A Sympy expression (the amplitude).
        n   : Number of external momenta p1, p2, ..., p_n.
        n_gluons, n_gravitons : # of gluons and gravitons, used for picking e_i indices.
                                If both are 0, only operation #3 is used (scalars).
        dim : (Optional) overall dimension, not strictly needed but included for consistency.
        only_scramble_numerator : If True, apply each scramble only to the numerator of 'amp'.
        number_of_scrambles     : How many times to apply a single scramble in succession.

        Returns:
        scrambled_amp: The final scrambled amplitude after 'number_of_scrambles' operations.
        """
        scrambled_amp = amp

        for _ in range(self.num_scrambles):
            if only_scramble_numerator:
                # Separate numerator and denominator
                num, den = sp.fraction(scrambled_amp)
                # Apply a single scramble to the numerator only
                new_num = self.single_scramble(num, n, n_gluons, n_gravitons)
                scrambled_amp = new_num / den
            else:
                # Scramble the entire expression
                scrambled_amp = self.single_scramble(scrambled_amp, n, n_gluons, n_gravitons)
        
        return scrambled_amp


## Test

Test the old generator + scrambler in terms of DP(...) dot products

In [158]:
exgenerator = ScatteringAmplitudeGenerator(max_particles=4, max_dim=6)
exmono = exgenerator.generate_monomial(4, 2, 0, 4)
exdeno = exgenerator.generate_denominator(4, 4)
exampl = exgenerator.generate_amplitude(4, 2, 0, 4)
print(exmono)
print(exdeno)
print(exampl)

DP(e2, e1)*DP(p1, p4)**2
DP(p1, p4)**2
DP(e2, e1)*DP(p1, p3)**2*DP(p3, p4)/DP(p3, p1)


Test the new generator + scrambler in terms of strings

In [161]:
exscrambler = ScatteringAmplitudeScrambler(num_scrambles=1)
exscrambledmono = exscrambler.scramble(exmono, 4, 2, 0, only_scramble_numerator=True)
exscrambledampl = exscrambler.scramble(exampl, 4, 2, 0, only_scramble_numerator=False)
print(f"Unscrambled: {exmono}, scrambled: {exscrambledmono}")
print(f"Unscrambled: {exampl}, scrambled: {exscrambledampl}")

Unscrambled: DP(e2, e1)*DP(p1, p4)**2, scrambled: (-DP(e2, p1) - DP(e2, p3) - DP(e2, p4))/DP(e2, p1) + DP(e2, e1)*DP(p1, p4)**2
Unscrambled: DP(e2, e1)*DP(p1, p3)**2*DP(p3, p4)/DP(p3, p1), scrambled: (-DP(e1, p1) - DP(e1, p2) - DP(e1, p4))*DP(e2, e1)*DP(p1, p3)**2*DP(p3, p4)/(DP(e1, p3)*DP(p3, p1))


In [171]:
newexgenerator = NewAmplitudeGenerator(max_particles=4, max_dim=6)
newexmono = newexgenerator.generate_monomial(4, 2, 2, 6)
newexdeno = newexgenerator.generate_denominator(4, 6)
newexampl = newexgenerator.generate_amplitude(4, 2, 2, 6)
print(newexmono)
print(newexdeno)
print(newexampl)

Duplicate in e_i.e_j: [1, 3, 4, 4] , shuffling.
e4.e3 * p4.p1 * e3.p1 * p4.p2 * e1.e4 * e2.p1
p1.p2 * p2.p3 * p4.p2
(e1.p3 * e3.p1 * e4.p3 * p3.p4 * e4.p3 * e2.p3 * e3.p4+e1.p3 * e4.p2 * e3.p1 * e2.p3 * e4.p1 * e3.p1 * p1.p4)/p2.p3


In [172]:
newexscrambler = NewAmplitudeScrambler(num_scrambles=1)
newexscrambledmono = newexscrambler.scramble(newexmono, 4, 2, 2, only_scramble_numerator=True)
newexscrambledampl = newexscrambler.scramble(newexampl, 4, 2, 2, only_scramble_numerator=False)
print(f"Unscrambled: {newexmono}, scrambled: {newexscrambledmono}")
print(f"Unscrambled: {newexampl}, scrambled: {newexscrambledampl}")

Unscrambled: e4.e3 * p4.p1 * e3.p1 * p4.p2 * e1.e4 * e2.p1, scrambled: (((e4.e3 * p4.p1 * e3.p1 * p4.p2 * e1.e4 * e2.p1)*((e4.(-(p1 + p2 + p3)))/(e4.p4))))/(1)
Unscrambled: (e1.p3 * e3.p1 * e4.p3 * p3.p4 * e4.p3 * e2.p3 * e3.p4+e1.p3 * e4.p2 * e3.p1 * e2.p3 * e4.p1 * e3.p1 * p1.p4)/p2.p3, scrambled: (((e1.p3 * e3.p1 * e4.p3 * p3.p4 * e4.p3 * e2.p3 * e3.p4+e1.p3 * e4.p2 * e3.p1 * e2.p3 * e4.p1 * e3.p1 * p1.p4)/p2.p3)*((e2.(-(p1 + p2 + p4)))/(e2.p3)))
